# **All Codes**

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
from google.colab import drive
from IPython.display import display # Import display function

# 2. Mount Google Drive and print a success message.
try:
    drive.mount('/content/drive', force_remount=True)
    print('Google Drive mounted successfully.')
except ValueError as e:
    print(f"Error mounting Google Drive: {e}")
    print("This often indicates an authentication problem or a temporary issue. Please try the following:")
    print("1. Restart the Colab runtime (Runtime -> Restart runtime...). This clears cached credentials.")
    print("2. After restarting, re-run this cell. Follow any authentication prompts and ensure you grant all necessary permissions.")
    print("3. If the issue persists and you'sre still seeing prompts about suspicious activity, you might need to check your Google account security settings directly.")

# 3. Define the zip_file_path and extraction_path, then unzip the football.zip file, listing its contents.
zip_file_path = '/content/drive/My Drive/Career/Practice for portfolio/football.zip'
extraction_path = 'football_data'
os.makedirs(extraction_path, exist_ok=True)

if os.path.exists(zip_file_path):
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extraction_path)
    print(f'File unzipped to: {extraction_path}')
    print('\nContents of the extracted directory:')
    for root, dirs, files in os.walk(extraction_path):
        level = root.replace(extraction_path, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')
else:
    print(f'File not found at: {zip_file_path}')
    print('Please ensure the path is correct and Drive is mounted.')

# 4. Load all relevant CSV files into pandas DataFrames
events_england_df = pd.read_csv('football_data/events_England.csv')
events_european_championship_df = pd.read_csv('football_data/events_European_Championship.csv')
teams_df = pd.read_csv('football_data/teams.csv')
print('\nDataFrames loaded: events_england_df, events_european_championship_df, teams_df')

# 5. Define the helper function get_zone(x_coord)
def get_zone(x_coord):
    if 0 <= x_coord <= 33:
        return 'Defensive'
    elif 34 <= x_coord <= 66:
        return 'Midfield'
    elif 67 <= x_coord <= 100:
        return 'Attacking'
    return 'Unknown' # Fallback for unexpected coordinates

# 6. Define the helper function euclidean_distance(p1, p2)
def euclidean_distance(p1, p2):
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

# 7. Define the helper function classify_possession_chain(chain_row, match_events_df, spain_team_id)
def classify_possession_chain(chain_row, match_events_df, spain_team_id):
    start_sec = chain_row['start_eventSec']
    end_sec = chain_row['end_eventSec']
    start_zone = chain_row['start_zone']
    end_zone = chain_row['end_zone']
    length_events = chain_row['length_events']

    if length_events < 3:
        return 'Ignored', 'Chain too short to be tactically meaningful.'

    chain_events = match_events_df[
        (match_events_df['eventSec'] >= start_sec) &
        (match_events_df['eventSec'] <= end_sec) &
        (match_events_df['teamId'] == spain_team_id)
    ].sort_values(by='eventSec').reset_index(drop=True)

    if chain_events.empty:
        return 'Unknown', 'No Spain events within designated chain timeframe.'

    first_event = chain_events.iloc[0]
    last_event = chain_events.iloc[-1]

    overall_x_progression = last_event['pos_dest_x'] - first_event['pos_orig_x']

    pass_events = chain_events[chain_events['eventName'] == 'Pass']
    avg_pass_x_change = 0
    if not pass_events.empty:
        avg_pass_x_change = (pass_events['pos_dest_x'] - pass_events['pos_orig_x']).mean()

    y_coords = pd.concat([chain_events['pos_orig_y'], chain_events['pos_dest_y']]).dropna()
    is_wing_focused = False
    if not y_coords.empty:
        on_wing_events_count = len(y_coords[(y_coords < 20) | (y_coords > 80)])
        is_wing_focused = (on_wing_events_count / len(y_coords)) > 0.6

    chain_duration = end_sec - start_sec
    if start_zone in ['Defensive', 'Midfield'] and end_zone == 'Attacking' and overall_x_progression > 45 and (chain_duration < 10 or length_events < 6):
        return 'Fast Transition', f"Rapid attack from {start_zone} to Attacking ({overall_x_progression:.1f} X prog) in {chain_duration:.1f}s / {length_events} events."

    max_x_achieved = chain_events['pos_dest_x'].max()
    if (max_x_achieved > 70 and end_zone in ['Defensive', 'Midfield'] and overall_x_progression < 10) or \
       (length_events > 1 and last_event['eventName'] == 'Pass' and (last_event['pos_dest_x'] - last_event['pos_orig_x']) < -15 and \
        chain_events['pos_dest_x'].max() > (first_event['pos_orig_x'] + 15)):
        return 'Reset / Recycle', f"Reached advanced area (max X {max_x_achieved:.1f}), then ended in {end_zone} with low net X progress ({overall_x_progression:.1f}) or ended with significant backward pass."

    if start_zone == 'Defensive' and overall_x_progression > 20 and avg_pass_x_change > 0 and length_events > 4:
        return 'Deep Build-up', f"Starts Defensive, strong overall X progression ({overall_x_progression:.1f}) with positive average pass progression ({avg_pass_x_change:.1f})."

    if overall_x_progression > 35 and avg_pass_x_change > 2 and length_events >= 3:
        return 'Direct Progression', f"Strong overall X progression ({overall_x_progression:.1f}) and positive average pass progression ({avg_pass_x_change:.1f})."

    if is_wing_focused and overall_x_progression > 10:
        return 'Wing Progression', f"Play predominantly along the wing ({len(y_coords[(y_coords < 20) | (y_coords > 80)])}/{len(y_coords)}) events) with forward progression ({overall_x_progression:.1f} X prog)."

    if start_zone in ['Midfield', 'Attacking'] and end_zone in ['Midfield', 'Attacking'] and overall_x_progression >= 0 and overall_x_progression <= 35 and length_events >= 3:
        return 'Probing Possession', f"Controlled possession in {start_zone}/{end_zone} with moderate X progression ({overall_x_progression:.1f}), searching for openings."

    return 'Circulation', f"Low overall X progression ({overall_x_progression:.1f}), possibly sideways/backward passes, indicating possession retention."

# 8. Define the helper function get_chain_events_sequence
def get_chain_events_sequence(chain_id_val, match_events_df_full, chain_info_row, selected_cols):
    start_sec = chain_info_row['start_eventSec']
    end_sec = chain_info_row['end_eventSec']

    chain_events = match_events_df_full[
        (match_events_df_full['eventSec'] >= start_sec) &
        (match_events_df_full['eventSec'] <= end_sec)
    ].sort_values(by='eventSec').copy()

    chain_events['chain_id'] = chain_id_val
    return chain_events[['chain_id'] + selected_cols]

# 9. Define the label_chain(features) function
def label_chain(features):
    THRESHOLDS = {
        'min_chain_events': 3,

        # Fast Transition
        'fast_trans_time_max': 10,
        'fast_trans_events_max': 6,
        'fast_trans_x_prog_per_sec_min': 5.0,
        'fast_trans_total_x_prog_min': 45,

        # Reset / Recycle — requires genuine backward movement, not just a backward pass
        'reset_recycle_max_x_reached_min': 70,
        'reset_recycle_total_x_prog_max': 10,
        'reset_recycle_backward_passes_min': 2,

        # Probing — lateral, patient, forward-oriented but not penetrating
        'probing_duration_min': 8,
        'probing_total_x_prog_min': 5,
        'probing_total_x_prog_max': 40,
        'probing_x_variance_min': 100,   # lateral movement expected
        'probing_directness_max': 0.5,   # meandering, not direct

        # Circulation — low progression but requires some structural evidence
        'circulation_total_x_prog_max': 15,
        'circulation_duration_min': 6,
        'circulation_backward_passes_min': 1,
    }

    total_x_progress = features['total_x_progress']
    max_x_reached = features['max_x_reached']
    time_duration = features['time_duration']
    events_count = features['length_events']
    x_progress_per_second = features['x_progress_per_second']
    num_backward_passes = features['num_backward_passes']
    last_event_is_backward_pass = features['last_event_is_backward_pass']
    start_zone = features['start_zone']
    end_zone = features['end_zone']
    x_variance = features['x_variance']
    directness_index = features['directness_index']

    # 1. Fast Transition — unchanged, seems solid
    if (start_zone in ['Defensive', 'Midfield'] and
        end_zone == 'Attacking' and
        total_x_progress > THRESHOLDS['fast_trans_total_x_prog_min'] and
        (time_duration < THRESHOLDS['fast_trans_time_max'] or
         events_count < THRESHOLDS['fast_trans_events_max']) and
        x_progress_per_second > THRESHOLDS['fast_trans_x_prog_per_sec_min']):
        return 'Fast Transition'

    # 2. Reset / Recycle — tightened: requires genuine regression, not just any backward pass
    if (max_x_reached > THRESHOLDS['reset_recycle_max_x_reached_min'] and
        end_zone in ['Defensive', 'Midfield'] and
        total_x_progress < THRESHOLDS['reset_recycle_total_x_prog_max'] and
        num_backward_passes >= THRESHOLDS['reset_recycle_backward_passes_min']):
        return 'Reset / Recycle'

    # 3. Probing Possession — lateral, patient, exploratory
    if (start_zone in ['Midfield', 'Attacking'] and
        end_zone in ['Midfield', 'Attacking'] and
        THRESHOLDS['probing_total_x_prog_min'] <= total_x_progress <= THRESHOLDS['probing_total_x_prog_max'] and
        time_duration >= THRESHOLDS['probing_duration_min'] and
        x_variance >= THRESHOLDS['probing_x_variance_min'] and
        directness_index <= THRESHOLDS['probing_directness_max']):
        return 'Probing Possession'

    # 4. Circulation — low progression but requires some structural evidence
    if (total_x_progress <= THRESHOLDS['circulation_total_x_prog_max'] and
        time_duration >= THRESHOLDS['circulation_duration_min'] and
        num_backward_passes >= THRESHOLDS['circulation_backward_passes_min']):
        return 'Circulation'

    return 'Undefined/Other'

# 10. Find the spain_team_id and set the match_id_to_analyze
spain_team = teams_df[teams_df['name'].str.contains('Spain', case=False, na=False)]
spain_team_id = spain_team['wyId'].iloc[0]
match_id_to_analyze = 1694409
print(f"\nSpain's teamId: {spain_team_id}, Sample Match ID: {match_id_to_analyze}")

# 11. Filter events for the specific match and sort by eventSec
match_events_df = events_european_championship_df[
    events_european_championship_df['matchId'] == match_id_to_analyze
].sort_values(by='eventSec').reset_index(drop=True)
print(f"Match events for match {match_id_to_analyze} filtered.")

# 12. Implement the logic to identify and extract Spain's possession chains
possession_chains_data = []
current_chain_events = []
chain_id_counter = 0
explicit_restart_events = ['Foul', 'Throw-in', 'Goal kick', 'Kick-off']

for index, event in match_events_df.iterrows():
    event_team_id = event['teamId']
    event_name = event['eventName']

    is_explicit_chain_breaker = (event_name in explicit_restart_events)

    if event_team_id == spain_team_id:
        if is_explicit_chain_breaker:
            if current_chain_events:
                chain_id_counter += 1
                first_event = current_chain_events[0]
                last_event = current_chain_events[-1]
                chain_data = {
                    'chain_id': chain_id_counter,
                    'start_eventSec': first_event['eventSec'],
                    'end_eventSec': last_event['eventSec'],
                    'length_events': len(current_chain_events),
                    'start_zone': get_zone(first_event['pos_orig_x']),
                    'end_zone': get_zone(last_event['pos_dest_x']),
                    'final_eventName': last_event['eventName'],
                    'has_shot': 1 if any(e['eventName'] == 'Shot' for e in current_chain_events) else 0
                }
                possession_chains_data.append(chain_data)
                current_chain_events = []
        else:
            current_chain_events.append(event)
    else:
        if current_chain_events:
            chain_id_counter += 1
            first_event = current_chain_events[0]
            last_event = current_chain_events[-1]
            chain_data = {
                'chain_id': chain_id_counter,
                'start_eventSec': first_event['eventSec'],
                'end_eventSec': last_event['eventSec'],
                'length_events': len(current_chain_events),
                'start_zone': get_zone(first_event['pos_orig_x']),
                'end_zone': get_zone(last_event['pos_dest_x']),
                'final_eventName': last_event['eventName'],
                'has_shot': 1 if any(e['eventName'] == 'Shot' for e in current_chain_events) else 0
            }
            possession_chains_data.append(chain_data)
            current_chain_events = []

if current_chain_events:
    chain_id_counter += 1
    first_event = current_chain_events[0]
    last_event = current_chain_events[-1]
    chain_data = {
        'chain_id': chain_id_counter,
        'start_eventSec': first_event['eventSec'],
        'end_eventSec': last_event['eventSec'],
        'length_events': len(current_chain_events),
        'start_zone': get_zone(first_event['pos_orig_x']),
        'end_zone': get_zone(last_event['pos_dest_x']),
        'final_eventName': last_event['eventName'],
        'has_shot': 1 if any(e['eventName'] == 'Shot' for e in current_chain_events) else 0
    }
    possession_chains_data.append(chain_data)

spain_possession_chains_df = pd.DataFrame(possession_chains_data)
print(f"\nSpain possession chains generated: {len(spain_possession_chains_df)}")

# 13. Filter spain_possession_chains_df to create meaningful_possession_chains_df
meaningful_possession_chains_df = spain_possession_chains_df[
    spain_possession_chains_df['length_events'] >= 3
].copy()
print(f"Meaningful possession chains (>=3 events) filtered: {len(meaningful_possession_chains_df)}")

# 14. Iterate through meaningful_possession_chains_df to engineer features
engineered_features = []
for index, chain_row in meaningful_possession_chains_df.iterrows():
    chain_id = int(chain_row['chain_id'])
    start_sec = chain_row['start_eventSec']
    end_sec = chain_row['end_eventSec']

    chain_events = match_events_df[
        (match_events_df['eventSec'] >= start_sec) &
        (match_events_df['eventSec'] <= end_sec) &
        (match_events_df['teamId'] == spain_team_id)
    ].sort_values(by='eventSec').reset_index(drop=True)

    if chain_events.empty:
        continue

    first_event = chain_events.iloc[0]
    last_event = chain_events.iloc[-1]

    total_x_progress = last_event['pos_dest_x'] - first_event['pos_orig_x']
    all_x_coords = pd.concat([chain_events['pos_orig_x'], chain_events['pos_dest_x']]).dropna()
    max_x_reached = all_x_coords.max()
    time_duration = end_sec - start_sec
    if time_duration == 0:
        time_duration = 0.01
    events_count = len(chain_events)
    x_progress_per_second = total_x_progress / time_duration
    backward_passes = chain_events[
        (chain_events['eventName'] == 'Pass') &
        (chain_events['pos_dest_x'] < chain_events['pos_orig_x'])
    ]
    num_backward_passes = len(backward_passes)
    last_event_is_backward_pass = False
    if last_event['eventName'] == 'Pass' and last_event['pos_dest_x'] < (last_event['pos_orig_x'] - 10):
        last_event_is_backward_pass = True
    start_zone_feat = get_zone(first_event['pos_orig_x'])
    end_zone_feat = get_zone(last_event['pos_dest_x'])
    x_variance = all_x_coords.var() if len(all_x_coords) > 1 else 0
    if np.isnan(x_variance): x_variance = 0

    start_point = (first_event['pos_orig_x'], first_event['pos_orig_y'])
    end_point = (last_event['pos_dest_x'], last_event['pos_dest_y'])
    straight_line_dist = euclidean_distance(start_point, end_point)

    path_length = 0
    for i in range(len(chain_events)):
        event = chain_events.iloc[i]
        path_length += euclidean_distance((event['pos_orig_x'], event['pos_orig_y']), (event['pos_dest_x'], event['pos_dest_y']))

    directness_index = straight_line_dist / path_length if path_length > 0 else 1.0

    engineered_features.append({
        'chain_id': chain_id,
        'start_eventSec': start_sec,
        'end_eventSec': end_sec,
        'length_events': events_count,
        'total_x_progress': total_x_progress,
        'max_x_reached': max_x_reached,
        'time_duration': time_duration,
        'x_progress_per_second': x_progress_per_second,
        'num_backward_passes': num_backward_passes,
        'last_event_is_backward_pass': last_event_is_backward_pass,
        'start_zone': start_zone_feat,
        'end_zone': end_zone_feat,
        'x_variance': x_variance,
        'directness_index': directness_index
    })

features_df = pd.DataFrame(engineered_features)
print(f"Features engineered for {len(features_df)} chains.")

# 15. Apply the classify_possession_chain function to generate tactical_archetypes_df
classified_chains_data = []
for index, row in spain_possession_chains_df.iterrows():
    archetype, reasoning = classify_possession_chain(row, match_events_df, spain_team_id)
    classified_chains_data.append({
        'chain_id': row['chain_id'],
        'length_events': row['length_events'],
        'start_zone': row['start_zone'],
        'end_zone': row['end_zone'],
        'primary_archetype': archetype,
        'short_reasoning': reasoning
    })

tactical_archetypes_df = pd.DataFrame(classified_chains_data)
print(f"\nInitial tactical archetypes classified for {len(tactical_archetypes_df)} chains.")

# 16. Apply the label_chain function to features_df
features_df['new_archetype_label'] = features_df.apply(label_chain, axis=1)
print("\nDistribution of new archetypes:")
print("Unique values in 'new_archetype_label':")
print(features_df['new_archetype_label'].unique())
print("\nNull values in 'new_archetype_label':", features_df['new_archetype_label'].isnull().sum())
display(features_df['new_archetype_label'].value_counts())
print("\nFeatures DataFrame with new labels (first 5 rows):")
display(features_df.head())

# 17. Filter tactical_archetypes_df to create chains_for_re_evaluation_df
chains_for_re_evaluation_df = tactical_archetypes_df[
    tactical_archetypes_df['primary_archetype'].isin(['Circulation', 'Probing Possession'])
].copy()
print(f"\nChains identified for re-evaluation ('Circulation' or 'Probing Possession'): {len(chains_for_re_evaluation_df)}")
display(chains_for_re_evaluation_df.head())

# 18. Simulate the manual re-evaluation process
sample_size = min(30, len(chains_for_re_evaluation_df))
sample_chain_ids = chains_for_re_evaluation_df['chain_id'].sample(n=sample_size, random_state=42).tolist()

dummy_data = {
    'chain_id': sample_chain_ids,
    'Current Label': np.random.choice(['Circulation', 'Probing Possession'], size=sample_size),
    'Re-evaluated Label': np.random.choice(['Circulation', 'Probing Possession', 'Direct Progression', 'Deep Build-up'], size=sample_size),
    'Tactical Reason': np.random.choice(['More progressive than thought', 'More conservative than thought', 'Clear attacking intent', 'Building up from deep', 'Ambiguous movement'], size=sample_size),
    'Confidence': np.random.choice(['High', 'Medium', 'Low'], size=sample_size, p=[0.6, 0.3, 0.1])
}
manual_re_evaluation_results_df = pd.DataFrame(dummy_data)
print("\nManual re-evaluation results (dummy data) simulated:")
display(manual_re_evaluation_results_df.head())

# 19. Merge chains_for_re_evaluation_df with manual_re_evaluation_results_df
merged_re_evaluation_df = pd.merge(
    chains_for_re_evaluation_df,
    manual_re_evaluation_results_df[['chain_id', 'Re-evaluated Label', 'Tactical Reason', 'Confidence']],
    on='chain_id',
    how='left'
)
merged_re_evaluation_df['Re-evaluated Label'] = merged_re_evaluation_df['Re-evaluated Label'].fillna(merged_re_evaluation_df['primary_archetype'])
merged_re_evaluation_df['Tactical Reason'] = merged_re_evaluation_df['Tactical Reason'].fillna('Not re-evaluated in sample')
merged_re_evaluation_df['Confidence'] = merged_re_evaluation_df['Confidence'].fillna('N/A')
print(f"\nMerged re-evaluation results created: {len(merged_re_evaluation_df)}")

# 20. Calculate the reclassification percentages
total_re_evaluated_chains = len(manual_re_evaluation_results_df)

circ_to_probing_reclass = merged_re_evaluation_df[
    (merged_re_evaluation_df['primary_archetype'] == 'Circulation') &
    (merged_re_evaluation_df['Re-evaluated Label'] == 'Probing Possession')
]
perc_circ_to_probing = (len(circ_to_probing_reclass) / total_re_evaluated_chains) * 100 if total_re_evaluated_chains > 0 else 0

probing_to_circ_reclass = merged_re_evaluation_df[
    (merged_re_evaluation_df['primary_archetype'] == 'Probing Possession') &
    (merged_re_evaluation_df['Re-evaluated Label'] == 'Circulation')
]
perc_probing_to_circ = (len(probing_to_circ_reclass) / total_re_evaluated_chains) * 100 if total_re_evaluated_chains > 0 else 0

# 21. Identify ambiguous chains
ambiguous_chains = merged_re_evaluation_df[
    (merged_re_evaluation_df['Confidence'] == 'Low')
].head(10)

# 22. Print a detailed summary of the manual re-evaluation
print("\n--- Manual Re-evaluation Summary ---")
print(f"Total chains processed for re-evaluation (sample): {total_re_evaluated_chains}")
print(f"Original 'Circulation' chains re-evaluated as 'Probing Possession': {len(circ_to_probing_reclass)} ({perc_circ_to_probing:.2f}%) ")
print(f"Original 'Probing Possession' chains re-evaluated as 'Circulation': {len(probing_to_circ_reclass)} ({perc_probing_to_circ:.2f}%) ")

print("\nTop 10 Ambiguous Chains (Confidence: Low):")
if not ambiguous_chains.empty:
    display(ambiguous_chains[[
        'chain_id', 'primary_archetype', 'Re-evaluated Label', 'Tactical Reason', 'Confidence'
    ]])
else:
    print("No ambiguous chains with 'Low' confidence found.")
print("--- End of Summary ---")

# 23. Finally, display the head of features_df, tactical_archetypes_df, and merged_re_evaluation_df.
print("\nHead of features_df:")
display(features_df.head())
print("\nHead of tactical_archetypes_df:")
display(tactical_archetypes_df.head())
print("\nHead of merged_re_evaluation_df:")
display(merged_re_evaluation_df.head())

Mounted at /content/drive
Google Drive mounted successfully.
File unzipped to: football_data

Contents of the extracted directory:
football_data/
    matches_Germany.csv
    actions.csv
    games.csv
    tags2name.csv
    events_European_Championship.csv
    coaches.csv
    matches_France.csv
    teams.csv
    labels.csv
    matches_World_Cup.csv
    events_World_Cup.csv
    events_Italy.csv
    matches_Italy.csv
    player_games.csv
    events_Spain.csv
    players.csv
    features.csv
    playerank.csv
    eventid2name.csv
    events_Germany.csv
    competitions.csv
    matches_European_Championship.csv
    matches_England.csv
    events_England.csv
    matches_Spain.csv
    referees.csv
    events_France.csv

DataFrames loaded: events_england_df, events_european_championship_df, teams_df

Spain's teamId: 1598, Sample Match ID: 1694409
Match events for match 1694409 filtered.

Spain possession chains generated: 313
Meaningful possession chains (>=3 events) filtered: 114
Features engi

,count
new_archetype_label,
Undefined/Other,47
Circulation,28
Reset / Recycle,22
Probing Possession,11
Fast Transition,6



Features DataFrame with new labels (first 5 rows):


,chain_id,start_eventSec,end_eventSec,length_events,total_x_progress,max_x_reached,time_duration,x_progress_per_second,num_backward_passes,last_event_is_backward_pass,start_zone,end_zone,x_variance,directness_index,new_archetype_label
0,4,23.914157,30.786905,3,-56,95,6.872748,-8.148124,1,False,Midfield,Defensive,1952.266667,0.770083,Circulation
1,5,32.717905,37.966041,5,-49,100,5.248136,-9.336648,2,False,Attacking,Midfield,697.877778,0.797004,Reset / Recycle
2,6,39.532623,55.226037,6,-22,76,15.693414,-1.401862,1,False,Midfield,Defensive,437.606061,1.276869,Circulation
3,8,58.658791,61.675848,3,-4,100,3.017057,-1.325795,1,False,Attacking,Attacking,1528.666667,0.116299,Undefined/Other
4,9,62.769868,78.684861,7,-15,70,15.914993,-0.942507,3,False,Attacking,Midfield,315.516484,0.231509,Circulation



Chains identified for re-evaluation ('Circulation' or 'Probing Possession'): 55


...

,chain_id,start_eventSec,end_eventSec,length_events,total_x_progress,max_x_reached,time_duration,x_progress_per_second,num_backward_passes,last_event_is_backward_pass,start_zone,end_zone,x_variance,directness_index,new_archetype_label
0,4,23.914157,30.786905,3,-56,95,6.872748,-8.148124,1,False,Midfield,Defensive,1952.266667,0.770083,Circulation
1,5,32.717905,37.966041,5,-49,100,5.248136,-9.336648,2,False,Attacking,Midfield,697.877778,0.797004,Reset / Recycle
2,6,39.532623,55.226037,6,-22,76,15.693414,-1.401862,1,False,Midfield,Defensive,437.606061,1.276869,Circulation
3,8,58.658791,61.675848,3,-4,100,3.017057,-1.325795,1,False,Attacking,Attacking,1528.666667,0.116299,Undefined/Other
4,9,62.769868,78.684861,7,-15,70,15.914993,-0.942507,3,False,Attacking,Midfield,315.516484,0.231509,Circulation



Head of tactical_archetypes_df:


,chain_id,length_events,start_zone,end_zone,primary_archetype,short_reasoning
0,1,1,Defensive,Midfield,Ignored,Chain too short to be tactically meaningful.
1,2,1,Midfield,Midfield,Ignored,Chain too short to be tactically meaningful.
2,3,1,Defensive,Defensive,Ignored,Chain too short to be tactically meaningful.
3,4,3,Midfield,Defensive,Reset / Recycle,"Reached advanced area (max X 95.0), then ended..."
4,5,5,Attacking,Midfield,Reset / Recycle,"Reached advanced area (max X 90.0), then ended..."



Head of merged_re_evaluation_df:


,chain_id,length_events,start_zone,end_zone,primary_archetype,short_reasoning,Re-evaluated Label,Tactical Reason,Confidence
0,8,3,Attacking,Attacking,Circulation,"Low overall X progression (-4.0), possibly sid...",Circulation,Not re-evaluated in sample,N/A
1,9,7,Attacking,Midfield,Circulation,"Low overall X progression (-15.0), possibly si...",Circulation,Not re-evaluated in sample,N/A
2,12,5,Defensive,Midfield,Circulation,"Low overall X progression (30.0), possibly sid...",Circulation,Not re-evaluated in sample,N/A
3,16,4,Attacking,Attacking,Probing Possession,Controlled possession in Attacking/Attacking w...,Circulation,Ambiguous movement,High
4,45,9,Attacking,Attacking,Probing Possession,Controlled possession in Attacking/Attacking w...,Deep Build-up,Clear attacking intent,Low


# **New Label Chain**

In [ ]:
# ── Apply revised label_chain to features_df ──────────────────────────────────

def label_chain_v2(features):
    THRESHOLDS = {
        # Fast Transition
        'fast_trans_time_max': 10,
        'fast_trans_events_max': 6,
        'fast_trans_x_prog_per_sec_min': 5.0,
        'fast_trans_total_x_prog_min': 45,

        # Reset / Recycle
        'reset_recycle_max_x_reached_min': 70,
        'reset_recycle_total_x_prog_max': 10,
        'reset_recycle_backward_passes_min': 2,

        # Probing Possession
        'probing_duration_min': 8,
        'probing_total_x_prog_min': 5,
        'probing_total_x_prog_max': 40,
        'probing_x_variance_min': 100,
        'probing_directness_max': 0.5,

        # Circulation
        'circulation_total_x_prog_max': 15,
        'circulation_duration_min': 6,
        'circulation_backward_passes_min': 1,
    }

    total_x_progress    = features['total_x_progress']
    max_x_reached       = features['max_x_reached']
    time_duration       = features['time_duration']
    events_count        = features['length_events']
    x_progress_per_sec  = features['x_progress_per_second']
    num_backward_passes = features['num_backward_passes']
    start_zone          = features['start_zone']
    end_zone            = features['end_zone']
    x_variance          = features['x_variance']
    directness_index    = features['directness_index']

    # 1. Fast Transition
    if (start_zone in ['Defensive', 'Midfield'] and
        end_zone == 'Attacking' and
        total_x_progress > THRESHOLDS['fast_trans_total_x_prog_min'] and
        (time_duration < THRESHOLDS['fast_trans_time_max'] or
         events_count < THRESHOLDS['fast_trans_events_max']) and
        x_progress_per_sec > THRESHOLDS['fast_trans_x_prog_per_sec_min']):
        return 'Fast Transition'

    # 2. Reset / Recycle
    if (max_x_reached > THRESHOLDS['reset_recycle_max_x_reached_min'] and
        end_zone in ['Defensive', 'Midfield'] and
        total_x_progress < THRESHOLDS['reset_recycle_total_x_prog_max'] and
        num_backward_passes >= THRESHOLDS['reset_recycle_backward_passes_min']):
        return 'Reset / Recycle'

    # 3. Probing Possession
    if (start_zone in ['Midfield', 'Attacking'] and
        end_zone in ['Midfield', 'Attacking'] and
        THRESHOLDS['probing_total_x_prog_min'] <= total_x_progress <= THRESHOLDS['probing_total_x_prog_max'] and
        time_duration >= THRESHOLDS['probing_duration_min'] and
        x_variance >= THRESHOLDS['probing_x_variance_min'] and
        directness_index <= THRESHOLDS['probing_directness_max']):
        return 'Probing Possession'

    # 4. Circulation
    if (total_x_progress <= THRESHOLDS['circulation_total_x_prog_max'] and
        time_duration >= THRESHOLDS['circulation_duration_min'] and
        num_backward_passes >= THRESHOLDS['circulation_backward_passes_min']):
        return 'Circulation'

    return 'Undefined/Other'


# ── Apply & compare ────────────────────────────────────────────────────────────

features_df['label_v2'] = features_df.apply(label_chain_v2, axis=1)

total = len(features_df)

print("=" * 52)
print("  ARCHETYPE DISTRIBUTION — REVISED RULES (v2)")
print("=" * 52)
counts_v2 = features_df['label_v2'].value_counts()
for label, count in counts_v2.items():
    bar = "█" * count
    print(f"  {label:<22} {count:>4}  ({count/total*100:>5.1f}%)  {bar}")
print(f"  {'TOTAL':<22} {total:>4}")

print()
print("=" * 52)
print("  COMPARISON — OLD vs NEW LABELS")
print("=" * 52)
comparison = features_df.groupby(['new_archetype_label', 'label_v2']).size().unstack(fill_value=0)
display(comparison)

print()
print("=" * 52)
print("  CHAINS THAT CHANGED LABEL")
print("=" * 52)
changed = features_df[features_df['new_archetype_label'] != features_df['label_v2']][
    ['chain_id', 'new_archetype_label', 'label_v2',
     'total_x_progress', 'time_duration', 'num_backward_passes',
     'x_variance', 'directness_index', 'start_zone', 'end_zone']
]
print(f"  {len(changed)} chains changed out of {total}\n")
display(changed)

  ARCHETYPE DISTRIBUTION — REVISED RULES (v2)
  Undefined/Other          47  ( 41.2%)  ███████████████████████████████████████████████
  Circulation              28  ( 24.6%)  ████████████████████████████
  Reset / Recycle          22  ( 19.3%)  ██████████████████████
  Probing Possession       11  (  9.6%)  ███████████
  Fast Transition           6  (  5.3%)  ██████
  TOTAL                   114

  COMPARISON — OLD vs NEW LABELS


label_v2,Circulation,Fast Transition,Probing Possession,Reset / Recycle,Undefined/Other
new_archetype_label,,,,,
Circulation,28,0,0,0,0
Fast Transition,0,6,0,0,0
Probing Possession,0,0,11,0,0
Reset / Recycle,0,0,0,22,0
Undefined/Other,0,0,0,0,47



  CHAINS THAT CHANGED LABEL
  0 chains changed out of 114



,chain_id,new_archetype_label,label_v2,total_x_progress,time_duration,num_backward_passes,x_variance,directness_index,start_zone,end_zone


# **Diagnosis for undefined chains**

In [ ]:
# ── Diagnose Undefined/Other chains ───────────────────────────────────────────

undefined_df = features_df[features_df['label_v2'] == 'Undefined/Other'].copy()

print(f"Undefined chains: {len(undefined_df)}\n")

print("=" * 52)
print("  ZONE COMBINATIONS")
print("=" * 52)
print(undefined_df.groupby(['start_zone', 'end_zone']).size().sort_values(ascending=False).to_string())

print()
print("=" * 52)
print("  KEY FEATURE RANGES")
print("=" * 52)
cols = ['total_x_progress', 'time_duration', 'num_backward_passes',
        'x_variance', 'directness_index', 'max_x_reached']
print(undefined_df[cols].describe().round(2).to_string())

print()
print("=" * 52)
print("  WHY DID EACH RULE FAIL?")
print("=" * 52)

def diagnose(row):
    reasons = []

    # Why not Circulation?
    if not (row['total_x_progress'] <= 15):
        reasons.append(f"Circ: x_prog too high ({row['total_x_progress']:.1f} > 15)")
    if not (row['time_duration'] >= 6):
        reasons.append(f"Circ: too short ({row['time_duration']:.1f}s < 6)")
    if not (row['num_backward_passes'] >= 1):
        reasons.append(f"Circ: no backward passes")

    # Why not Probing?
    if not (row['start_zone'] in ['Midfield', 'Attacking']):
        reasons.append(f"Prob: wrong start zone ({row['start_zone']})")
    if not (5 <= row['total_x_progress'] <= 40):
        reasons.append(f"Prob: x_prog out of range ({row['total_x_progress']:.1f})")
    if not (row['time_duration'] >= 8):
        reasons.append(f"Prob: too short ({row['time_duration']:.1f}s < 8)")
    if not (row['x_variance'] >= 100):
        reasons.append(f"Prob: low x_variance ({row['x_variance']:.1f} < 100)")
    if not (row['directness_index'] <= 0.5):
        reasons.append(f"Prob: too direct ({row['directness_index']:.2f} > 0.5)")

    return " | ".join(reasons)

undefined_df['fail_reasons'] = undefined_df.apply(diagnose, axis=1)

# Count most common failure reasons
from collections import Counter
all_reasons = []
for r in undefined_df['fail_reasons']:
    all_reasons.extend(r.split(' | '))

print("\nMost common reasons chains fall through:\n")
for reason, count in Counter(all_reasons).most_common():
    print(f"  {count:>3}x  {reason}")

Undefined chains: 47

  ZONE COMBINATIONS
start_zone  end_zone 
Defensive   Midfield     10
Midfield    Attacking     9
Attacking   Attacking     6
            Defensive     5
Midfield    Defensive     5
            Midfield      5
Defensive   Attacking     3
            Defensive     3
Attacking   Midfield      1

  KEY FEATURE RANGES
       total_x_progress  time_duration  num_backward_passes  x_variance  directness_index  max_x_reached
count             47.00          47.00                47.00       47.00             47.00          47.00
mean               7.70           6.61                 1.04      727.78              0.64          80.62
std               36.33           6.02                 1.22      609.61              0.53          16.57
min              -94.00           1.14                 0.00       11.47              0.00          36.00
25%               -5.00           2.85                 0.00      263.82              0.26          69.50
50%                8.00         

Chain Diagnosis

In [ ]:
# ── Diagnostic: verify chain counts and filtering rates ───────────────────────

print("=" * 65)
print("  CHAIN COUNT DIAGNOSTIC — SAMPLE TEAMS")
print("=" * 65)

# Pick a few contrasting teams to inspect
check_teams = {
    'Spain': spain_team_id,
}

# Also find Iceland and Croatia IDs
for name in ['Iceland', 'Croatia', 'Republic of Ireland']:
    row = teams_df[teams_df['name'].str.contains(name, case=False, na=False)]
    if not row.empty:
        check_teams[name] = row['wyId'].iloc[0]

for team_name, team_id in check_teams.items():
    team_match_ids = matches_ec[
        matches_ec['team_ids'].apply(
            lambda ids: str(team_id) in [str(i) for i in ids]
        )
    ]['wyId'].tolist()

    print(f"\n{team_name} (ID: {team_id}) — {len(team_match_ids)} matches")
    print(f"  {'Match':<12} {'Raw chains':>12} {'>=3 events':>12} {'Filtered out':>12} {'Filter rate':>12}")
    print(f"  {'-'*12} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")

    for match_id in team_match_ids:
        match_events = events_european_championship_df[
            events_european_championship_df['matchId'] == match_id
        ].sort_values('eventSec').reset_index(drop=True)

        # Count raw chains
        chains_data = []
        current_chain = []
        chain_counter = 0
        explicit_breaks = ['Foul', 'Throw-in', 'Goal kick', 'Kick-off']

        for _, event in match_events.iterrows():
            is_break = event['eventName'] in explicit_breaks
            if event['teamId'] == team_id:
                if is_break:
                    if current_chain:
                        chain_counter += 1
                        chains_data.append(len(current_chain))
                        current_chain = []
                else:
                    current_chain.append(event)
            else:
                if current_chain:
                    chain_counter += 1
                    chains_data.append(len(current_chain))
                    current_chain = []

        if current_chain:
            chain_counter += 1
            chains_data.append(len(current_chain))

        raw_total = len(chains_data)
        meaningful = sum(1 for x in chains_data if x >= 3)
        filtered = raw_total - meaningful
        filter_rate = (filtered / raw_total * 100) if raw_total > 0 else 0

        print(f"  {match_id:<12} {raw_total:>12} {meaningful:>12} {filtered:>12} {filter_rate:>11.1f}%")

    print()
    print(f"  Event type breakdown for last match ({team_match_ids[-1]}):")
    sample_events = events_european_championship_df[
        (events_european_championship_df['matchId'] == team_match_ids[-1]) &
        (events_european_championship_df['teamId'] == team_id)
    ]
    print(f"  Total team events: {len(sample_events)}")
    print(f"  Event types:")
    for evt, cnt in sample_events['eventName'].value_counts().head(8).items():
        print(f"    {evt:<25} {cnt:>4}")

  CHAIN COUNT DIAGNOSTIC — SAMPLE TEAMS

Spain (ID: 1598) — 4 matches
  Match          Raw chains   >=3 events Filtered out  Filter rate
  ------------ ------------ ------------ ------------ ------------
  1694432               353           92          261        73.9%
  1694412               336          112          224        66.7%
  1694411               388           90          298        76.8%
  1694409               313          114          199        63.6%

  Event type breakdown for last match (1694409):
  Total team events: 1013
  Event types:
    Pass                       687
    Duel                       191
    Free Kick                   55
    Others on the ball          43
    Shot                        15
    Foul                        11
    Offside                      7
    Save attempt                 3

Iceland (ID: 7839) — 5 matches
  Match          Raw chains   >=3 events Filtered out  Filter rate
  ------------ ------------ ------------ ------------ ----

# **Label Chain V3**

In [ ]:
def label_chain_v3(features):
    THRESHOLDS = {
        # Fast Transition
        'fast_trans_time_max': 10,
        'fast_trans_events_max': 6,
        'fast_trans_x_prog_per_sec_min': 5.0,
        'fast_trans_total_x_prog_min': 45,

        # Reset / Recycle
        'reset_recycle_max_x_reached_min': 70,
        'reset_recycle_total_x_prog_max': 10,
        'reset_recycle_backward_passes_min': 2,

        # Probing Possession
        'probing_duration_min': 6,
        'probing_total_x_prog_min': 5,
        'probing_total_x_prog_max': 40,
        'probing_x_variance_min': 100,
        'probing_directness_max': 0.7,

        # Circulation
        'circulation_total_x_prog_max': 15,
        'circulation_duration_min': 4,

        # Short Possession
        'short_possession_duration_max': 4,
    }

    total_x_progress    = features['total_x_progress']
    max_x_reached       = features['max_x_reached']
    time_duration       = features['time_duration']
    events_count        = features['length_events']
    x_progress_per_sec  = features['x_progress_per_second']
    num_backward_passes = features['num_backward_passes']
    start_zone          = features['start_zone']
    end_zone            = features['end_zone']
    x_variance          = features['x_variance']
    directness_index    = features['directness_index']

    # 1. Fast Transition
    if (start_zone in ['Defensive', 'Midfield'] and
        end_zone == 'Attacking' and
        total_x_progress > THRESHOLDS['fast_trans_total_x_prog_min'] and
        (time_duration < THRESHOLDS['fast_trans_time_max'] or
         events_count < THRESHOLDS['fast_trans_events_max']) and
        x_progress_per_sec > THRESHOLDS['fast_trans_x_prog_per_sec_min']):
        return 'Fast Transition'

    # 2. Reset / Recycle
    if (max_x_reached > THRESHOLDS['reset_recycle_max_x_reached_min'] and
        end_zone in ['Defensive', 'Midfield'] and
        total_x_progress < THRESHOLDS['reset_recycle_total_x_prog_max'] and
        num_backward_passes >= THRESHOLDS['reset_recycle_backward_passes_min']):
        return 'Reset / Recycle'

    # 3. Probing Possession
    # All zones allowed — Spain probes from deep too
    if (end_zone in ['Midfield', 'Attacking'] and
        THRESHOLDS['probing_total_x_prog_min'] <= total_x_progress <= THRESHOLDS['probing_total_x_prog_max'] and
        time_duration >= THRESHOLDS['probing_duration_min'] and
        x_variance >= THRESHOLDS['probing_x_variance_min'] and
        directness_index <= THRESHOLDS['probing_directness_max']):
        return 'Probing Possession'

    # 4. Circulation
    # No backward pass requirement — lateral/sideways possession counts too
    if (total_x_progress <= THRESHOLDS['circulation_total_x_prog_max'] and
        time_duration >= THRESHOLDS['circulation_duration_min']):
        return 'Circulation'

    # 5. Short Possession
    # Brief sequences with limited tactical structure
    if time_duration < THRESHOLDS['short_possession_duration_max']:
        return 'Short Possession'

    return 'Undefined/Other'


# ── Apply & display ────────────────────────────────────────────────────────────

features_df['label_v3'] = features_df.apply(label_chain_v3, axis=1)

total = len(features_df)

print("=" * 55)
print("  ARCHETYPE DISTRIBUTION — v3")
print("=" * 55)
counts_v3 = features_df['label_v3'].value_counts()
for label, count in counts_v3.items():
    bar = "█" * count
    print(f"  {label:<25} {count:>4}  ({count/total*100:>5.1f}%)  {bar}")
print(f"  {'TOTAL':<25} {total:>4}")

print()
print("=" * 55)
print("  v2 → v3 CHANGES")
print("=" * 55)
comparison = features_df.groupby(['label_v2', 'label_v3']).size().unstack(fill_value=0)
display(comparison)

print()
changed = features_df[features_df['label_v2'] != features_df['label_v3']]
print(f"  {len(changed)} chains changed label from v2 to v3")

  ARCHETYPE DISTRIBUTION — v3
  Circulation                 37  ( 32.5%)  █████████████████████████████████████
  Reset / Recycle             22  ( 19.3%)  ██████████████████████
  Short Possession            18  ( 15.8%)  ██████████████████
  Probing Possession          18  ( 15.8%)  ██████████████████
  Undefined/Other             13  ( 11.4%)  █████████████
  Fast Transition              6  (  5.3%)  ██████
  TOTAL                      114

  v2 → v3 CHANGES


label_v3,Circulation,Fast Transition,Probing Possession,Reset / Recycle,Short Possession,Undefined/Other
label_v2,,,,,,
Circulation,27,0,1,0,0,0
Fast Transition,0,6,0,0,0,0
Probing Possession,0,0,11,0,0,0
Reset / Recycle,0,0,0,22,0,0
Undefined/Other,10,0,6,0,18,13



  35 chains changed label from v2 to v3


Checking New Undefineds

In [ ]:
# ── Diagnose remaining Undefined/Other in v3 ──────────────────────────────────

undefined_v3 = features_df[features_df['label_v3'] == 'Undefined/Other'].copy()

def diagnose_v3(row):
    reasons = []

    if not (row['total_x_progress'] <= 15 and row['time_duration'] >= 4):
        if row['total_x_progress'] > 15:
            reasons.append(f"Circ: x_prog too high ({row['total_x_progress']:.1f})")
        if row['time_duration'] < 4:
            reasons.append(f"Circ: too short ({row['time_duration']:.1f}s)")

    if not (5 <= row['total_x_progress'] <= 40):
        reasons.append(f"Prob: x_prog out of range ({row['total_x_progress']:.1f})")
    if not (row['x_variance'] >= 100):
        reasons.append(f"Prob: low x_variance ({row['x_variance']:.1f})")
    if not (row['directness_index'] <= 0.7):
        reasons.append(f"Prob: too direct ({row['directness_index']:.2f})")
    if not (row['time_duration'] >= 6):
        reasons.append(f"Prob: too short ({row['time_duration']:.1f}s)")

    return " | ".join(reasons) if reasons else "unknown"

undefined_v3['fail_reasons'] = undefined_v3.apply(diagnose_v3, axis=1)

print(f"Remaining Undefined chains: {len(undefined_v3)}\n")
print("=" * 55)
print("  FEATURE SUMMARY")
print("=" * 55)
cols = ['total_x_progress', 'time_duration', 'num_backward_passes',
        'x_variance', 'directness_index', 'max_x_reached', 'start_zone', 'end_zone']
display(undefined_v3[cols + ['fail_reasons']])

Remaining Undefined chains: 13

  FEATURE SUMMARY


,total_x_progress,time_duration,num_backward_passes,x_variance,directness_index,max_x_reached,start_zone,end_zone,fail_reasons
5,30,13.532697,2,554.266667,2.279162,69,Defensive,Midfield,Circ: x_prog too high (30.0) | Prob: too direc...
6,25,5.516555,1,1765.839286,0.277756,100,Attacking,Attacking,Circ: x_prog too high (25.0) | Prob: too short...
7,23,7.535371,0,454.300000,0.663388,68,Defensive,Defensive,Circ: x_prog too high (23.0)
10,60,13.659748,3,480.450216,0.261675,99,Midfield,Attacking,Circ: x_prog too high (60.0) | Prob: x_prog ou...
17,30,5.998350,2,1376.219780,0.196930,100,Defensive,Midfield,Circ: x_prog too high (30.0) | Prob: too short...
31,43,27.831351,5,530.108696,0.221347,96,Midfield,Attacking,Circ: x_prog too high (43.0) | Prob: x_prog ou...
45,57,7.467845,0,1434.410714,0.822674,91,Defensive,Midfield,Circ: x_prog too high (57.0) | Prob: x_prog ou...
47,29,4.087099,1,718.711111,0.674719,96,Attacking,Attacking,Circ: x_prog too high (29.0) | Prob: too short...
60,55,4.712910,1,605.866667,1.026472,72,Defensive,Midfield,Circ: x_prog too high (55.0) | Prob: x_prog ou...
75,35,9.649871,0,93.410714,0.869032,81,Midfield,Attacking,Circ: x_prog too high (35.0) | Prob: low x_var...


Label Chain V4

In [ ]:
def label_chain_v4(features):
    THRESHOLDS = {
        # Fast Transition
        'fast_trans_time_max': 10,
        'fast_trans_events_max': 6,
        'fast_trans_x_prog_per_sec_min': 5.0,
        'fast_trans_total_x_prog_min': 45,

        # Reset / Recycle
        'reset_recycle_max_x_reached_min': 70,
        'reset_recycle_total_x_prog_max': 10,
        'reset_recycle_backward_passes_min': 2,
        'reset_recycle_deep_x_reached_min': 80,   # deep penetration threshold

        # Direct Progression
        'direct_prog_x_prog_min': 30,
        'direct_prog_duration_min': 5,
        'direct_prog_directness_max': 0.90,

        # Probing Possession
        'probing_duration_min': 6,
        'probing_total_x_prog_min': 5,
        'probing_total_x_prog_max': 40,
        'probing_x_variance_min': 100,
        'probing_directness_max': 0.75,

        # Circulation
        'circulation_total_x_prog_max': 25,
        'circulation_duration_min': 4,

        # Short Possession
        'short_possession_duration_max': 5,
    }

    total_x_progress    = features['total_x_progress']
    max_x_reached       = features['max_x_reached']
    time_duration       = features['time_duration']
    events_count        = features['length_events']
    x_progress_per_sec  = features['x_progress_per_second']
    num_backward_passes = features['num_backward_passes']
    start_zone          = features['start_zone']
    end_zone            = features['end_zone']
    x_variance          = features['x_variance']
    directness_index    = features['directness_index']

    # 1. Fast Transition
    if (start_zone in ['Defensive', 'Midfield'] and
        end_zone == 'Attacking' and
        total_x_progress > THRESHOLDS['fast_trans_total_x_prog_min'] and
        (time_duration < THRESHOLDS['fast_trans_time_max'] or
         events_count < THRESHOLDS['fast_trans_events_max']) and
        x_progress_per_sec > THRESHOLDS['fast_trans_x_prog_per_sec_min']):
        return 'Fast Transition'

    # 2. Reset / Recycle
    # Either: advanced + multiple backward passes
    # Or: penetrated very deep + any backward movement
    if (max_x_reached > THRESHOLDS['reset_recycle_max_x_reached_min'] and
        end_zone in ['Defensive', 'Midfield'] and
        total_x_progress < THRESHOLDS['reset_recycle_total_x_prog_max'] and
        (num_backward_passes >= THRESHOLDS['reset_recycle_backward_passes_min'] or
         (max_x_reached > THRESHOLDS['reset_recycle_deep_x_reached_min'] and
          total_x_progress < 0))):
        return 'Reset / Recycle'

    # 3. Direct Progression
    if (total_x_progress >= THRESHOLDS['direct_prog_x_prog_min'] and
        time_duration >= THRESHOLDS['direct_prog_duration_min'] and
        directness_index <= THRESHOLDS['direct_prog_directness_max'] and
        end_zone in ['Midfield', 'Attacking']):
        return 'Direct Progression'

    # 4. Probing Possession
    if (end_zone in ['Midfield', 'Attacking'] and
        THRESHOLDS['probing_total_x_prog_min'] <= total_x_progress <= THRESHOLDS['probing_total_x_prog_max'] and
        time_duration >= THRESHOLDS['probing_duration_min'] and
        x_variance >= THRESHOLDS['probing_x_variance_min'] and
        directness_index <= THRESHOLDS['probing_directness_max']):
        return 'Probing Possession'

    # 5. Circulation
    if (total_x_progress <= THRESHOLDS['circulation_total_x_prog_max'] and
        time_duration >= THRESHOLDS['circulation_duration_min']):
        return 'Circulation'

    # 5b. Circulation fallback — directness > 1.0 indicates data artifact
    # treat as low-quality possession, classify by end zone and progress
    if (directness_index > 1.0 and
        total_x_progress <= 35 and
        end_zone in ['Defensive', 'Midfield']):
        return 'Circulation'

    # 6. Short Possession
    if time_duration < THRESHOLDS['short_possession_duration_max']:
        return 'Short Possession'

    return 'Undefined/Other'


# ── Apply & display ────────────────────────────────────────────────────────────

features_df['label_v4'] = features_df.apply(label_chain_v4, axis=1)

total = len(features_df)

print("=" * 55)
print("  ARCHETYPE DISTRIBUTION — v4 (updated Reset fix)")
print("=" * 55)
counts_v4 = features_df['label_v4'].value_counts()
for label, count in counts_v4.items():
    bar = "█" * count
    print(f"  {label:<25} {count:>4}  ({count/total*100:>5.1f}%)  {bar})")
print(f"  {'TOTAL':<25} {total:>4}")

print()
print("=" * 55)
print("  CHAINS THAT MOVED TO RESET / RECYCLE")
print("=" * 55)
moved = features_df[
    (features_df['label_v4'] == 'Reset / Recycle') &
    (features_df['label_v3'] != 'Reset / Recycle')
][['chain_id', 'total_x_progress', 'max_x_reached',
   'num_backward_passes', 'start_zone', 'end_zone', 'label_v3']]
print(f"  {len(moved)} chains newly classified as Reset / Recycle\n")
display(moved)

  ARCHETYPE DISTRIBUTION — v4 (updated Reset fix)
  Reset / Recycle             35  ( 30.7%)  ███████████████████████████████████)
  Circulation                 34  ( 29.8%)  ██████████████████████████████████)
  Short Possession            14  ( 12.3%)  ██████████████)
  Probing Possession          14  ( 12.3%)  ██████████████)
  Direct Progression          11  (  9.6%)  ███████████)
  Fast Transition              6  (  5.3%)  ██████)
  TOTAL                      114

  CHAINS THAT MOVED TO RESET / RECYCLE
  13 chains newly classified as Reset / Recycle



,chain_id,total_x_progress,max_x_reached,num_backward_passes,start_zone,end_zone,label_v3
0,4,-56,95,1,Midfield,Defensive,Circulation
19,64,-34,82,1,Attacking,Midfield,Circulation
30,92,-3,92,1,Midfield,Midfield,Short Possession
51,158,-23,84,1,Attacking,Midfield,Short Possession
67,196,-4,100,1,Defensive,Defensive,Short Possession
73,208,-56,84,1,Attacking,Defensive,Circulation
81,229,-13,92,1,Attacking,Midfield,Circulation
92,259,-27,88,1,Midfield,Midfield,Circulation
98,271,-66,88,1,Midfield,Defensive,Circulation
99,272,-92,100,0,Attacking,Defensive,Short Possession


In [ ]:
undefined_v4 = features_df[features_df['label_v4'] == 'Undefined/Other']
display(undefined_v4[['chain_id', 'total_x_progress', 'time_duration',
                       'max_x_reached', 'x_variance', 'directness_index',
                       'start_zone', 'end_zone', 'num_backward_passes']])

,chain_id,total_x_progress,time_duration,max_x_reached,x_variance,directness_index,start_zone,end_zone,num_backward_passes


Chains Inspector

In [ ]:
# ── See available chain IDs per archetype ─────────────────────────────────────
print("Available chain IDs by archetype:\n")
for archetype in features_df['label_v4'].value_counts().index:
    ids = features_df[features_df['label_v4'] == archetype]['chain_id'].tolist()
    print(f"  {archetype:<25} {ids}")

Available chain IDs by archetype:

  Reset / Recycle           [4, 5, 34, 48, 64, 76, 89, 92, 95, 126, 158, 184, 186, 194, 196, 208, 227, 229, 231, 235, 241, 243, 247, 252, 259, 266, 270, 271, 272, 281, 284, 287, 296, 302, 304]
  Circulation               [6, 9, 12, 16, 25, 50, 54, 62, 66, 80, 82, 94, 97, 100, 102, 106, 112, 113, 134, 170, 175, 176, 192, 199, 206, 221, 262, 263, 265, 283, 289, 293, 294, 309]
  Short Possession          [8, 69, 86, 129, 149, 179, 182, 183, 198, 214, 223, 234, 246, 277]
  Probing Possession        [45, 47, 49, 72, 83, 104, 109, 110, 124, 178, 195, 219, 226, 230]
  Direct Progression        [42, 61, 93, 101, 125, 169, 193, 216, 233, 276, 292]
  Fast Transition           [32, 73, 151, 161, 202, 207]


In [ ]:
# ── Full Archetype Inspector ───────────────────────────────────────────────────

def inspect_all_archetypes(features_df, match_events_df, spain_team_id):

    archetype_order = [
        'Fast Transition',
        'Direct Progression',
        'Probing Possession',
        'Circulation',
        'Reset / Recycle',
        'Short Possession',
        'Undefined/Other'
    ]

    def get_event_sequence(chain_meta):
        start_sec = chain_meta['start_eventSec']
        end_sec   = chain_meta['end_eventSec']
        return match_events_df[
            (match_events_df['eventSec'] >= start_sec) &
            (match_events_df['eventSec'] <= end_sec) &
            (match_events_df['teamId'] == spain_team_id)
        ].sort_values(by='eventSec').reset_index(drop=True)

    def draw_pitch(chain_events):
        pitch = [['·'] * 50 for _ in range(10)]
        for i, row in chain_events.iterrows():
            orig_x = row.get('pos_orig_x', None)
            orig_y = row.get('pos_orig_y', None)
            dest_x = row.get('pos_dest_x', None)
            dest_y = row.get('pos_dest_y', None)
            if pd.notna(orig_x) and pd.notna(orig_y):
                px = min(int(orig_x / 100 * 49), 49)
                py = min(int(orig_y / 100 * 9), 9)
                pitch[py][px] = 'o'
            if pd.notna(dest_x) and pd.notna(dest_y):
                px = min(int(dest_x / 100 * 49), 49)
                py = min(int(dest_y / 100 * 9), 9)
                if i == len(chain_events) - 1:
                    pitch[py][px] = 'X'
                else:
                    pitch[py][px] = '+'
        for row in pitch:
            row[16] = '|'
            row[33] = '|'
        return pitch

    def print_chain(chain_meta, chain_events, chain_num, total):
        chain_id = int(chain_meta['chain_id'])
        label    = chain_meta['label_v4']

        print(f"\n  Chain {chain_id}  ({chain_num}/{total})")
        print(f"  {'-'*55}")
        print(f"  Duration     : {chain_meta['time_duration']:.1f}s   "
              f"Events: {int(chain_meta['length_events'])}")
        print(f"  Zones        : {chain_meta['start_zone']} → {chain_meta['end_zone']}")
        print(f"  X progress   : {chain_meta['total_x_progress']:.1f}   "
              f"Max X: {chain_meta['max_x_reached']:.1f}")
        print(f"  Directness   : {chain_meta['directness_index']:.2f}   "
              f"X variance: {chain_meta['x_variance']:.1f}")
        print(f"  Backward pass: {int(chain_meta['num_backward_passes'])}")
        print()

        # Event sequence
        print(f"  {'#':<4} {'Event':<22} {'From':>10}  {'To':>10}  {'ΔX':>6}  {'Dir'}")
        print(f"  {'-'*4} {'-'*22} {'-'*10}  {'-'*10}  {'-'*6}  {'-'*10}")
        for i, row in chain_events.iterrows():
            orig_x = row.get('pos_orig_x', None)
            orig_y = row.get('pos_orig_y', None)
            dest_x = row.get('pos_dest_x', None)
            dest_y = row.get('pos_dest_y', None)
            from_str  = f"({orig_x:.0f},{orig_y:.0f})" if pd.notna(orig_x) and pd.notna(orig_y) else "—"
            to_str    = f"({dest_x:.0f},{dest_y:.0f})" if pd.notna(dest_x) and pd.notna(dest_y) else "—"
            if pd.notna(orig_x) and pd.notna(dest_x):
                delta_x   = dest_x - orig_x
                direction = "→ fwd" if delta_x > 3 else ("← back" if delta_x < -3 else "↔ lat")
                delta_str = f"{delta_x:>+.0f}"
            else:
                delta_str = "—"
                direction = ""
            event_name = str(row.get('eventName', ''))[:22]
            print(f"  {i+1:<4} {event_name:<22} {from_str:>10}  {to_str:>10}  {delta_str:>6}  {direction}")

        # Pitch map
        print()
        print(f"  OWN GOAL      DEF  |      MID      |  ATT    OPP GOAL")
        for row in draw_pitch(chain_events):
            print("  " + "".join(row))
        print(f"  o=origin  +=destination  X=final destination")

    # ── Main loop ─────────────────────────────────────────────────────────────
    for archetype in archetype_order:
        archetype_chains = features_df[
            features_df['label_v4'] == archetype
        ].sort_values('chain_id').reset_index(drop=True)

        if archetype_chains.empty:
            continue

        total = len(archetype_chains)

        print("\n" + "=" * 65)
        print(f"  ARCHETYPE: {archetype.upper()}  ({total} chains)")
        print("=" * 65)

        for chain_num, (_, chain_meta) in enumerate(archetype_chains.iterrows(), start=1):
            chain_events = get_event_sequence(chain_meta)
            if chain_events.empty:
                print(f"\n  Chain {int(chain_meta['chain_id'])} — no events found, skipping.")
                continue
            print_chain(chain_meta, chain_events, chain_num, total)

        print(f"\n  END OF {archetype.upper()}")
        print("=" * 65)


# ── Run ───────────────────────────────────────────────────────────────────────
inspect_all_archetypes(features_df, match_events_df, spain_team_id)


  ARCHETYPE: FAST TRANSITION  (6 chains)

  Chain 32  (1/6)
  -------------------------------------------------------
  Duration     : 1.8s   Events: 4
  Zones        : Defensive → Attacking
  X progress   : 77.0   Max X: 84.0
  Directness   : 1.27   X variance: 1149.1
  Backward pass: 1

  #    Event                        From          To      ΔX  Dir
  ---- ---------------------- ----------  ----------  ------  ----------
  1    Pass                       (6,55)     (45,26)     +39  → fwd
  2    Free Kick                 (71,27)     (84,20)     +13  → fwd
  3    Pass                      (16,80)     (17,79)      +1  ↔ lat
  4    Pass                      (84,20)     (83,21)      -1  ↔ lat

  OWN GOAL      DEF  |      MID      |  ATT    OPP GOAL
  ················|················|················
  ················|················|······Xo········
  ················|·····+··········|o···············
  ················|················|················
  ··o·············|··········

# **Testing in 4 Spain Matches**

In [ ]:
# ── Run v4 classification across 10 Spain matches ─────────────────────────────

import pandas as pd
import numpy as np

# ── Reuse helpers from your existing code ─────────────────────────────────────

def get_zone(x_coord):
    if 0 <= x_coord <= 33:
        return 'Defensive'
    elif 34 <= x_coord <= 66:
        return 'Midfield'
    elif 67 <= x_coord <= 100:
        return 'Attacking'
    return 'Unknown'

def euclidean_distance(p1, p2):
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def label_chain_v4(features):
    THRESHOLDS = {
        'fast_trans_time_max': 10,
        'fast_trans_events_max': 6,
        'fast_trans_x_prog_per_sec_min': 5.0,
        'fast_trans_total_x_prog_min': 45,
        'reset_recycle_max_x_reached_min': 70,
        'reset_recycle_total_x_prog_max': 10,
        'reset_recycle_backward_passes_min': 2,
        'reset_recycle_deep_x_reached_min': 80,
        'direct_prog_x_prog_min': 30,
        'direct_prog_duration_min': 5,
        'direct_prog_directness_max': 0.90,
        'probing_duration_min': 6,
        'probing_total_x_prog_min': 5,
        'probing_total_x_prog_max': 40,
        'probing_x_variance_min': 100,
        'probing_directness_max': 0.75,
        'circulation_total_x_prog_max': 25,
        'circulation_duration_min': 4,
        'short_possession_duration_max': 5,
    }

    total_x_progress    = features['total_x_progress']
    max_x_reached       = features['max_x_reached']
    time_duration       = features['time_duration']
    events_count        = features['length_events']
    x_progress_per_sec  = features['x_progress_per_second']
    num_backward_passes = features['num_backward_passes']
    start_zone          = features['start_zone']
    end_zone            = features['end_zone']
    x_variance          = features['x_variance']
    directness_index    = features['directness_index']

    if (start_zone in ['Defensive', 'Midfield'] and
        end_zone == 'Attacking' and
        total_x_progress > THRESHOLDS['fast_trans_total_x_prog_min'] and
        (time_duration < THRESHOLDS['fast_trans_time_max'] or
         events_count < THRESHOLDS['fast_trans_events_max']) and
        x_progress_per_sec > THRESHOLDS['fast_trans_x_prog_per_sec_min']):
        return 'Fast Transition'

    if (max_x_reached > THRESHOLDS['reset_recycle_max_x_reached_min'] and
        end_zone in ['Defensive', 'Midfield'] and
        total_x_progress < THRESHOLDS['reset_recycle_total_x_prog_max'] and
        (num_backward_passes >= THRESHOLDS['reset_recycle_backward_passes_min'] or
         (max_x_reached > THRESHOLDS['reset_recycle_deep_x_reached_min'] and
          total_x_progress < 0))):
        return 'Reset / Recycle'

    if (total_x_progress >= THRESHOLDS['direct_prog_x_prog_min'] and
        time_duration >= THRESHOLDS['direct_prog_duration_min'] and
        directness_index <= THRESHOLDS['direct_prog_directness_max'] and
        end_zone in ['Midfield', 'Attacking']):
        return 'Direct Progression'

    if (end_zone in ['Midfield', 'Attacking'] and
        THRESHOLDS['probing_total_x_prog_min'] <= total_x_progress <= THRESHOLDS['probing_total_x_prog_max'] and
        time_duration >= THRESHOLDS['probing_duration_min'] and
        x_variance >= THRESHOLDS['probing_x_variance_min'] and
        directness_index <= THRESHOLDS['probing_directness_max']):
        return 'Probing Possession'

    if (total_x_progress <= THRESHOLDS['circulation_total_x_prog_max'] and
        time_duration >= THRESHOLDS['circulation_duration_min']):
        return 'Circulation'

    if (directness_index > 1.0 and
        total_x_progress <= 35 and
        end_zone in ['Defensive', 'Midfield']):
        return 'Circulation'

    if time_duration < THRESHOLDS['short_possession_duration_max']:
        return 'Short Possession'

    return 'Undefined/Other'


def process_match(match_id, events_df, spain_team_id):
    """Build chains, engineer features, classify — for one match."""

    match_events = events_df[
        events_df['matchId'] == match_id
    ].sort_values('eventSec').reset_index(drop=True)

    if match_events.empty:
        return pd.DataFrame()

    # ── Build possession chains ───────────────────────────────────────────────
    chains_data = []
    current_chain = []
    chain_counter = 0
    explicit_breaks = ['Foul', 'Throw-in', 'Goal kick', 'Kick-off']

    for _, event in match_events.iterrows():
        is_break = event['eventName'] in explicit_breaks
        if event['teamId'] == spain_team_id:
            if is_break:
                if current_chain:
                    chain_counter += 1
                    fe, le = current_chain[0], current_chain[-1]
                    chains_data.append({
                        'chain_id': chain_counter,
                        'match_id': match_id,
                        'start_eventSec': fe['eventSec'],
                        'end_eventSec': le['eventSec'],
                        'length_events': len(current_chain),
                        'start_zone': get_zone(fe['pos_orig_x']),
                        'end_zone': get_zone(le['pos_dest_x']),
                    })
                    current_chain = []
            else:
                current_chain.append(event)
        else:
            if current_chain:
                chain_counter += 1
                fe, le = current_chain[0], current_chain[-1]
                chains_data.append({
                    'chain_id': chain_counter,
                    'match_id': match_id,
                    'start_eventSec': fe['eventSec'],
                    'end_eventSec': le['eventSec'],
                    'length_events': len(current_chain),
                    'start_zone': get_zone(fe['pos_orig_x']),
                    'end_zone': get_zone(le['pos_dest_x']),
                })
                current_chain = []

    if current_chain:
        chain_counter += 1
        fe, le = current_chain[0], current_chain[-1]
        chains_data.append({
            'chain_id': chain_counter,
            'match_id': match_id,
            'start_eventSec': fe['eventSec'],
            'end_eventSec': le['eventSec'],
            'length_events': len(current_chain),
            'start_zone': get_zone(fe['pos_orig_x']),
            'end_zone': get_zone(le['pos_dest_x']),
        })

    if not chains_data:
        return pd.DataFrame()

    chains_df = pd.DataFrame(chains_data)
    meaningful = chains_df[chains_df['length_events'] >= 3].copy()

    if meaningful.empty:
        return pd.DataFrame()

    # ── Engineer features ─────────────────────────────────────────────────────
    engineered = []
    for _, row in meaningful.iterrows():
        chain_events = match_events[
            (match_events['eventSec'] >= row['start_eventSec']) &
            (match_events['eventSec'] <= row['end_eventSec']) &
            (match_events['teamId'] == spain_team_id)
        ].sort_values('eventSec').reset_index(drop=True)

        if chain_events.empty:
            continue

        fe = chain_events.iloc[0]
        le = chain_events.iloc[-1]

        total_x_progress = le['pos_dest_x'] - fe['pos_orig_x']
        all_x = pd.concat([chain_events['pos_orig_x'], chain_events['pos_dest_x']]).dropna()
        max_x_reached = all_x.max() if not all_x.empty else 0
        time_duration = row['end_eventSec'] - row['start_eventSec']
        if time_duration == 0:
            time_duration = 0.01
        events_count = len(chain_events)
        x_progress_per_second = total_x_progress / time_duration

        backward_passes = chain_events[
            (chain_events['eventName'] == 'Pass') &
            (chain_events['pos_dest_x'] < chain_events['pos_orig_x'])
        ]
        num_backward_passes = len(backward_passes)

        x_variance = all_x.var() if len(all_x) > 1 else 0
        if np.isnan(x_variance):
            x_variance = 0

        start_point = (fe['pos_orig_x'], fe['pos_orig_y'])
        end_point   = (le['pos_dest_x'], le['pos_dest_y'])
        straight    = euclidean_distance(start_point, end_point)
        path_length = sum(
            euclidean_distance(
                (chain_events.iloc[i]['pos_orig_x'], chain_events.iloc[i]['pos_orig_y']),
                (chain_events.iloc[i]['pos_dest_x'], chain_events.iloc[i]['pos_dest_y'])
            )
            for i in range(len(chain_events))
        )
        directness_index = straight / path_length if path_length > 0 else 1.0

        engineered.append({
            'chain_id': row['chain_id'],
            'match_id': match_id,
            'length_events': events_count,
            'total_x_progress': total_x_progress,
            'max_x_reached': max_x_reached,
            'time_duration': time_duration,
            'x_progress_per_second': x_progress_per_second,
            'num_backward_passes': num_backward_passes,
            'start_zone': row['start_zone'],
            'end_zone': row['end_zone'],
            'x_variance': x_variance,
            'directness_index': directness_index,
        })

    if not engineered:
        return pd.DataFrame()

    feat_df = pd.DataFrame(engineered)
    feat_df['archetype'] = feat_df.apply(label_chain_v4, axis=1)
    return feat_df


# ── Find 10 Spain matches ──────────────────────────────────────────────────────

matches_ec = pd.read_csv('football_data/matches_European_Championship.csv')

spain_matches = matches_ec[
    (matches_ec['teamsData'].str.contains(str(spain_team_id), na=False))
].head(10)

match_ids = spain_matches['wyId'].tolist()
print(f"Found {len(match_ids)} Spain matches: {match_ids}\n")

# ── Process all matches ────────────────────────────────────────────────────────

all_results = []

for match_id in match_ids:
    print(f"Processing match {match_id}...", end=' ')
    feat_df = process_match(match_id, events_european_championship_df, spain_team_id)
    if feat_df.empty:
        print("no data")
        continue
    all_results.append(feat_df)
    print(f"{len(feat_df)} chains classified")

# ── Build summary table ────────────────────────────────────────────────────────

archetypes = [
    'Fast Transition',
    'Direct Progression',
    'Probing Possession',
    'Circulation',
    'Reset / Recycle',
    'Short Possession',
    'Undefined/Other',
]

rows = []
for feat_df in all_results:
    match_id = feat_df['match_id'].iloc[0]
    total = len(feat_df)
    counts = feat_df['archetype'].value_counts()
    row = {'Match ID': match_id, 'Total Chains': total}
    for arch in archetypes:
        n = counts.get(arch, 0)
        row[arch] = f"{n} ({n/total*100:.0f}%)"
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Match ID')

print("\n" + "=" * 100)
print("  SPAIN — ARCHETYPE DISTRIBUTION ACROSS 10 MATCHES")
print("=" * 100)
display(summary_df)

# ── Raw counts version for easier comparison ───────────────────────────────────
print("\n" + "=" * 100)
print("  RAW COUNTS")
print("=" * 100)

rows_raw = []
for feat_df in all_results:
    match_id = feat_df['match_id'].iloc[0]
    total = len(feat_df)
    counts = feat_df['archetype'].value_counts()
    row = {'Match ID': match_id, 'Total': total}
    for arch in archetypes:
        row[arch] = counts.get(arch, 0)
    rows_raw.append(row)

summary_raw = pd.DataFrame(rows_raw).set_index('Match ID')
display(summary_raw)

# ── Averages row ───────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  AVERAGES ACROSS ALL MATCHES")
print("=" * 60)
numeric_cols = ['Total'] + archetypes
avg = summary_raw[numeric_cols].mean().round(1)
for col, val in avg.items():
    if col == 'Total':
        print(f"  {'Total chains':<25} {val}")
    else:
        pct = (val / avg['Total'] * 100) if avg['Total'] > 0 else 0
        print(f"  {col:<25} {val:>5}  ({pct:.1f}%)")

Found 4 Spain matches: [1694432, 1694412, 1694411, 1694409]

Processing match 1694432... 92 chains classified
Processing match 1694412... 112 chains classified
Processing match 1694411... 90 chains classified
Processing match 1694409... 114 chains classified

  SPAIN — ARCHETYPE DISTRIBUTION ACROSS 10 MATCHES


,Total Chains,Fast Transition,Direct Progression,Probing Possession,Circulation,Reset / Recycle,Short Possession,Undefined/Other
Match ID,,,,,,,,
1694432,92,14 (15%),11 (12%),13 (14%),21 (23%),22 (24%),10 (11%),1 (1%)
1694412,112,6 (5%),9 (8%),10 (9%),32 (29%),32 (29%),22 (20%),1 (1%)
1694411,90,2 (2%),10 (11%),7 (8%),26 (29%),30 (33%),13 (14%),2 (2%)
1694409,114,6 (5%),11 (10%),14 (12%),34 (30%),35 (31%),14 (12%),0 (0%)



  RAW COUNTS


,Total,Fast Transition,Direct Progression,Probing Possession,Circulation,Reset / Recycle,Short Possession,Undefined/Other
Match ID,,,,,,,,
1694432,92,14,11,13,21,22,10,1
1694412,112,6,9,10,32,32,22,1
1694411,90,2,10,7,26,30,13,2
1694409,114,6,11,14,34,35,14,0



  AVERAGES ACROSS ALL MATCHES
  Total chains              102.0
  Fast Transition             7.0  (6.9%)
  Direct Progression         10.2  (10.0%)
  Probing Possession         11.0  (10.8%)
  Circulation                28.2  (27.6%)
  Reset / Recycle            29.8  (29.2%)
  Short Possession           14.8  (14.5%)
  Undefined/Other             1.0  (1.0%)


Testing in all European tournament matches

In [ ]:
# ── Full Euro Tournament — All Teams, All Matches ─────────────────────────────

import pandas as pd
import numpy as np
import ast

# ── Parse teamsData to get both team IDs per match ────────────────────────────

def get_team_ids_from_match(teams_data_str):
    try:
        data = ast.literal_eval(teams_data_str)
        return list(data.keys())
    except:
        return []

# ── Build full team list from matches ─────────────────────────────────────────

matches_ec['team_ids'] = matches_ec['teamsData'].apply(get_team_ids_from_match)

all_team_ids = set()
for ids in matches_ec['team_ids']:
    all_team_ids.update([int(i) for i in ids])

print(f"Total teams found in tournament: {len(all_team_ids)}")

# Map team IDs to names
team_id_to_name = dict(zip(teams_df['wyId'], teams_df['name']))

# ── Process one match for any team ────────────────────────────────────────────

def process_match_for_team(match_id, events_df, team_id):
    match_events = events_df[
        events_df['matchId'] == match_id
    ].sort_values('eventSec').reset_index(drop=True)

    if match_events.empty:
        return pd.DataFrame()

    chains_data = []
    current_chain = []
    chain_counter = 0
    explicit_breaks = ['Foul', 'Throw-in', 'Goal kick', 'Kick-off']

    for _, event in match_events.iterrows():
        is_break = event['eventName'] in explicit_breaks
        if event['teamId'] == team_id:
            if is_break:
                if current_chain:
                    chain_counter += 1
                    fe, le = current_chain[0], current_chain[-1]
                    chains_data.append({
                        'chain_id': chain_counter,
                        'match_id': match_id,
                        'start_eventSec': fe['eventSec'],
                        'end_eventSec': le['eventSec'],
                        'length_events': len(current_chain),
                        'start_zone': get_zone(fe['pos_orig_x']),
                        'end_zone': get_zone(le['pos_dest_x']),
                    })
                    current_chain = []
            else:
                current_chain.append(event)
        else:
            if current_chain:
                chain_counter += 1
                fe, le = current_chain[0], current_chain[-1]
                chains_data.append({
                    'chain_id': chain_counter,
                    'match_id': match_id,
                    'start_eventSec': fe['eventSec'],
                    'end_eventSec': le['eventSec'],
                    'length_events': len(current_chain),
                    'start_zone': get_zone(fe['pos_orig_x']),
                    'end_zone': get_zone(le['pos_dest_x']),
                })
                current_chain = []

    if current_chain:
        chain_counter += 1
        fe, le = current_chain[0], current_chain[-1]
        chains_data.append({
            'chain_id': chain_counter,
            'match_id': match_id,
            'start_eventSec': fe['eventSec'],
            'end_eventSec': le['eventSec'],
            'length_events': len(current_chain),
            'start_zone': get_zone(fe['pos_orig_x']),
            'end_zone': get_zone(le['pos_dest_x']),
        })

    if not chains_data:
        return pd.DataFrame()

    chains_df = pd.DataFrame(chains_data)
    meaningful = chains_df[chains_df['length_events'] >= 3].copy()
    if meaningful.empty:
        return pd.DataFrame()

    engineered = []
    for _, row in meaningful.iterrows():
        chain_events = match_events[
            (match_events['eventSec'] >= row['start_eventSec']) &
            (match_events['eventSec'] <= row['end_eventSec']) &
            (match_events['teamId'] == team_id)
        ].sort_values('eventSec').reset_index(drop=True)

        if chain_events.empty:
            continue

        fe = chain_events.iloc[0]
        le = chain_events.iloc[-1]

        total_x_progress = le['pos_dest_x'] - fe['pos_orig_x']
        all_x = pd.concat([chain_events['pos_orig_x'], chain_events['pos_dest_x']]).dropna()
        max_x_reached = all_x.max() if not all_x.empty else 0
        time_duration = row['end_eventSec'] - row['start_eventSec']
        if time_duration == 0:
            time_duration = 0.01
        events_count = len(chain_events)
        x_progress_per_second = total_x_progress / time_duration

        backward_passes = chain_events[
            (chain_events['eventName'] == 'Pass') &
            (chain_events['pos_dest_x'] < chain_events['pos_orig_x'])
        ]
        num_backward_passes = len(backward_passes)

        x_variance = all_x.var() if len(all_x) > 1 else 0
        if np.isnan(x_variance): x_variance = 0

        start_point = (fe['pos_orig_x'], fe['pos_orig_y'])
        end_point   = (le['pos_dest_x'], le['pos_dest_y'])
        straight    = euclidean_distance(start_point, end_point)
        path_length = sum(
            euclidean_distance(
                (chain_events.iloc[i]['pos_orig_x'], chain_events.iloc[i]['pos_orig_y']),
                (chain_events.iloc[i]['pos_dest_x'], chain_events.iloc[i]['pos_dest_y'])
            )
            for i in range(len(chain_events))
        )
        directness_index = straight / path_length if path_length > 0 else 1.0

        engineered.append({
            'chain_id': row['chain_id'],
            'match_id': match_id,
            'length_events': events_count,
            'total_x_progress': total_x_progress,
            'max_x_reached': max_x_reached,
            'time_duration': time_duration,
            'x_progress_per_second': x_progress_per_second,
            'num_backward_passes': num_backward_passes,
            'start_zone': row['start_zone'],
            'end_zone': row['end_zone'],
            'x_variance': x_variance,
            'directness_index': directness_index,
        })

    if not engineered:
        return pd.DataFrame()

    feat_df = pd.DataFrame(engineered)
    feat_df['archetype'] = feat_df.apply(label_chain_v4, axis=1)
    return feat_df


# ── Archetype columns ──────────────────────────────────────────────────────────

ARCHETYPES = [
    'Fast Transition',
    'Direct Progression',
    'Probing Possession',
    'Circulation',
    'Reset / Recycle',
    'Short Possession',
    'Undefined/Other',
]

SHORT = {
    'Fast Transition':    'FT%',
    'Direct Progression': 'DP%',
    'Probing Possession': 'Prob%',
    'Circulation':        'Circ%',
    'Reset / Recycle':    'Reset%',
    'Short Possession':   'Short%',
    'Undefined/Other':    'Undef%',
}

# ── Build and display one table per team ──────────────────────────────────────

team_summary = {}   # store avg vectors for cross-team comparison later

for team_id in sorted(all_team_ids):
    team_name = team_id_to_name.get(team_id, f"Team {team_id}")

    # find this team's matches
    team_match_ids = matches_ec[
        matches_ec['team_ids'].apply(lambda ids: str(team_id) in [str(i) for i in ids])
    ]['wyId'].tolist()

    if not team_match_ids:
        continue

    rows = []
    for m_num, match_id in enumerate(team_match_ids, start=1):
        feat_df = process_match_for_team(
            match_id, events_european_championship_df, team_id
        )
        if feat_df.empty:
            continue

        total = len(feat_df)
        counts = feat_df['archetype'].value_counts()
        row = {'Match': f"Match {m_num}", 'Chains': total}
        for arch in ARCHETYPES:
            n = counts.get(arch, 0)
            row[SHORT[arch]] = f"{n/total*100:.0f}%"
        rows.append(row)

    if not rows:
        continue

    team_df = pd.DataFrame(rows).set_index('Match')

    # averages row
    numeric_rows = []
    for feat_df_row in rows:
        numeric_rows.append({
            k: float(v.replace('%','')) for k, v in feat_df_row.items()
            if k not in ['Match', 'Chains']
        })
    avg_row = pd.DataFrame(numeric_rows).mean().round(1)
    avg_series = pd.Series(
        {'Chains': ''} | {k: f"{v}%" for k, v in avg_row.items()},
        name='AVG'
    )
    team_df = pd.concat([team_df, avg_series.to_frame().T])

    team_summary[team_name] = avg_row.to_dict()

    print(f"\n{'='*75}")
    print(f"  {team_name.upper()}")
    print(f"{'='*75}")
    display(team_df)


# ── Cross-team comparison table ───────────────────────────────────────────────

print(f"\n{'='*75}")
print(f"  TOURNAMENT COMPARISON — ALL TEAMS (average % per archetype)")
print(f"{'='*75}")

comparison_df = pd.DataFrame(team_summary).T
comparison_df.columns = [SHORT[a] for a in ARCHETYPES]
comparison_df = comparison_df.sort_values('Circ%', ascending=False)
comparison_df = comparison_df.round(1)

display(comparison_df)

Total teams found in tournament: 24

  SPAIN


,Chains,FT%,DP%,Prob%,Circ%,Reset%,Short%,Undef%
Match 1,92,15%,12%,14%,23%,24%,11%,1%
Match 2,112,5%,8%,9%,29%,29%,20%,1%
Match 3,90,2%,11%,8%,29%,33%,14%,2%
Match 4,114,5%,10%,12%,30%,31%,12%,0%
AVG,,6.8%,10.2%,10.8%,27.8%,29.2%,14.2%,1.0%



  ENGLAND


,Chains,FT%,DP%,Prob%,Circ%,Reset%,Short%,Undef%
Match 1,114,4%,7%,8%,35%,22%,23%,1%
Match 2,89,2%,7%,10%,19%,42%,13%,7%
Match 3,98,7%,7%,14%,24%,22%,23%,1%
Match 4,79,9%,4%,11%,32%,28%,15%,1%
AVG,,5.5%,6.2%,10.8%,27.5%,28.5%,18.5%,2.5%



  GERMANY


...

,Chains,FT%,DP%,Prob%,Circ%,Reset%,Short%,Undef%
Match 1,59,7%,3%,8%,25%,27%,27%,2%
Match 2,38,0%,13%,18%,24%,24%,21%,0%
Match 3,57,5%,7%,9%,28%,16%,35%,0%
Match 4,88,5%,5%,12%,31%,27%,18%,2%
AVG,,4.2%,7.0%,11.8%,27.0%,23.5%,25.2%,1.0%



  UKRAINE


,Chains,FT%,DP%,Prob%,Circ%,Reset%,Short%,Undef%
Match 1,109,3%,2%,8%,28%,28%,29%,3%
Match 2,101,9%,6%,15%,20%,31%,18%,2%
Match 3,49,8%,8%,4%,31%,20%,27%,2%
AVG,,6.7%,5.3%,9.0%,26.3%,26.3%,24.7%,2.3%



  TOURNAMENT COMPARISON — ALL TEAMS (average % per archetype)


,FT%,DP%,Prob%,Circ%,Reset%,Short%,Undef%
Romania,3.7,6.0,3.3,37.0,26.0,21.7,3.0
Albania,5.0,4.0,4.7,30.3,29.7,21.0,5.7
Wales,4.5,6.3,8.0,29.7,26.8,23.5,1.3
Russia,5.0,8.3,8.0,29.3,22.0,25.7,1.0
Hungary,7.0,6.8,6.5,29.0,28.2,21.5,1.0
Northern Ireland,3.8,4.2,7.2,28.5,29.2,26.2,1.0
France,6.6,4.6,5.1,27.9,29.1,24.3,2.1
Spain,6.8,10.2,10.8,27.8,29.2,14.2,1.0
Iceland,4.4,7.2,3.6,27.8,25.2,30.2,1.6
Belgium,5.6,7.4,8.6,27.8,24.6,24.4,2.4


# **NEW: Chains/90 + percentage combined comparison**

In [ ]:
# ── Updated comparison with per-90 rates ──────────────────────────────────────

print("=" * 80)
print("  TOURNAMENT COMPARISON — COUNTS PER 90 MIN + PERCENTAGES")
print("=" * 80)

# Rebuild team_summary with both raw counts and match counts for per-90
team_summary_full = {}

for team_id in sorted(all_team_ids):
    team_name = team_id_to_name.get(team_id, f"Team {team_id}")

    team_match_ids = matches_ec[
        matches_ec['team_ids'].apply(
            lambda ids: str(team_id) in [str(i) for i in ids]
        )
    ]['wyId'].tolist()

    if not team_match_ids:
        continue

    all_feat_dfs = []
    for match_id in team_match_ids:
        feat_df = process_match_for_team(
            match_id, events_european_championship_df, team_id
        )
        if not feat_df.empty:
            all_feat_dfs.append(feat_df)

    if not all_feat_dfs:
        continue

    combined = pd.concat(all_feat_dfs, ignore_index=True)
    total_chains = len(combined)
    num_matches = len(all_feat_dfs)
    chains_per_90 = total_chains / num_matches
    counts = combined['archetype'].value_counts()

    row = {
        'Matches': num_matches,
        'Chains/90': round(chains_per_90, 1),
    }
    for arch in ARCHETYPES:
        n = counts.get(arch, 0)
        per90 = n / num_matches
        pct = n / total_chains * 100
        row[f"{SHORT[arch]}"] = f"{per90:.1f} ({pct:.0f}%)"

    team_summary_full[team_name] = row

summary_full_df = pd.DataFrame(team_summary_full).T
summary_full_df = summary_full_df.sort_values('Chains/90', ascending=False)

display(summary_full_df)

# ── Also print a clean numeric version sorted by Chains/90 ────────────────────
print("\n" + "=" * 65)
print("  CHAINS PER 90 — RAW RANKING")
print("  (how many meaningful possession sequences per match)")
print("=" * 65)

team_chains = {}
for team_id in sorted(all_team_ids):
    team_name = team_id_to_name.get(team_id, f"Team {team_id}")
    team_match_ids = matches_ec[
        matches_ec['team_ids'].apply(
            lambda ids: str(team_id) in [str(i) for i in ids]
        )
    ]['wyId'].tolist()

    total = 0
    matches = 0
    for match_id in team_match_ids:
        feat_df = process_match_for_team(
            match_id, events_european_championship_df, team_id
        )
        if not feat_df.empty:
            total += len(feat_df)
            matches += 1

    if matches > 0:
        team_chains[team_name] = round(total / matches, 1)

for team, c90 in sorted(team_chains.items(), key=lambda x: -x[1]):
    bar = "█" * int(c90 / 3)
    print(f"  {team:<25} {c90:>6}  {bar}")

  TOURNAMENT COMPARISON — COUNTS PER 90 MIN + PERCENTAGES


,Matches,Chains/90,FT%,DP%,Prob%,Circ%,Reset%,Short%,Undef%
Germany,6,107.5,8.3 (8%),9.0 (8%),11.3 (11%),26.0 (24%),28.8 (27%),21.0 (20%),3.0 (3%)
Spain,4,102.0,7.0 (7%),10.2 (10%),11.0 (11%),28.2 (28%),29.8 (29%),14.8 (14%),1.0 (1%)
England,4,95.0,5.2 (6%),6.0 (6%),10.2 (11%),26.5 (28%),26.5 (28%),18.2 (19%),2.2 (2%)
France,7,94.7,5.7 (6%),4.4 (5%),4.9 (5%),26.6 (28%),27.1 (29%),23.9 (25%),2.1 (2%)
Portugal,7,93.1,6.1 (7%),6.7 (7%),7.1 (8%),20.6 (22%),26.0 (28%),24.9 (27%),1.7 (2%)
Switzerland,4,92.2,4.0 (4%),4.2 (5%),4.0 (4%),23.8 (26%),32.5 (35%),22.2 (24%),1.5 (2%)
Ukraine,3,86.3,5.3 (6%),4.0 (5%),8.7 (10%),21.7 (25%),23.7 (27%),21.0 (24%),2.0 (2%)
Austria,3,84.7,5.0 (6%),5.3 (6%),4.0 (5%),23.7 (28%),21.3 (25%),25.0 (30%),0.3 (0%)
Belgium,5,84.4,4.8 (6%),6.2 (7%),7.0 (8%),23.0 (27%),20.8 (25%),20.4 (24%),2.2 (3%)
Sweden,3,80.3,3.7 (5%),4.0 (5%),7.3 (9%),19.7 (24%),25.0 (31%),18.3 (23%),2.3 (3%)



  CHAINS PER 90 — RAW RANKING
  (how many meaningful possession sequences per match)
  Germany                    107.5  ███████████████████████████████████
  Spain                      102.0  ██████████████████████████████████
  England                     95.0  ███████████████████████████████
  France                      94.7  ███████████████████████████████
  Portugal                    93.1  ███████████████████████████████
  Switzerland                 92.2  ██████████████████████████████
  Ukraine                     86.3  ████████████████████████████
  Austria                     84.7  ████████████████████████████
  Belgium                     84.4  ████████████████████████████
  Sweden                      80.3  ██████████████████████████
  Croatia                     80.0  ██████████████████████████
  Poland                      79.6  ██████████████████████████
  Hungary                     74.8  ████████████████████████
  Russia                      73.7  ███████████████████